## Datos útiles
- Kilometraje
- Costo Viaje (Dinero o consumo combustible)
- Presión neumáticos inicio viaje
- Presión neumático fin viaje
- Presencia de grietas
- Profundidad surcos de neumáticos

In [1]:
"""
Generate a synthetic dataset of truck trips for tire cost/efficiency modeling.

Each row = one trip, with:
- trip_id: unique identifier
- cost_usd: total cost of the trip
- distance_km: distance traveled
- tread_depth_mm: tire tread depth measured for the trip (avg across tires)
- has_cracks: whether the tires show visible cracks (True/False)
- pressure_start_psi: tire pressure at the start of the trip
- pressure_end_psi: tire pressure at the end of the trip
"""

import numpy as np
import pandas as pd

# Reproducibility
RNG = np.random.default_rng(seed=42)
N_TRIPS = 2000

# ---- Core trip variables ----
trip_id = np.arange(1, N_TRIPS + 1)

# Distance per trip (km) - right-skewed: mostly short/medium hauls, some long hauls
distance_km = np.round(RNG.gamma(shape=3.0, scale=80.0, size=N_TRIPS) + 20, 1)
distance_km = np.clip(distance_km, 20, 1500)

# Cost of trip (USD): base rate per km + fuel/toll noise + fixed overhead per trip
cost_per_km = RNG.normal(loc=0.85, scale=0.12, size=N_TRIPS)  # $/km, varies by route/fuel price
fixed_overhead = RNG.normal(loc=25, scale=5, size=N_TRIPS)    # tolls, loading fees, etc.
cost_usd = np.round(distance_km * cost_per_km + fixed_overhead, 2)
cost_usd = np.clip(cost_usd, 10, None)

# Tread depth (mm): new tires ~18mm, legal minimum ~1.6mm
tread_depth_mm = np.round(RNG.uniform(1.5, 18.0, size=N_TRIPS), 2)

# Probability of cracks increases as tread depth decreases (older/worn tires)
crack_prob = np.clip(0.55 - (tread_depth_mm / 18.0) * 0.5, 0.03, 0.6)
has_cracks = RNG.random(N_TRIPS) < crack_prob

# Tire pressure (psi): typical heavy truck tires run ~100-120 psi cold
pressure_start_psi = np.round(RNG.normal(loc=110, scale=6, size=N_TRIPS), 1)

# Pressure change over the trip: slight increase from heat build-up over distance,
# plus random noise, plus a bigger drop if tires have cracks (slow leak)
heat_gain = distance_km * RNG.uniform(0.005, 0.02, size=N_TRIPS)
leak_loss = np.where(has_cracks, RNG.uniform(2, 8, size=N_TRIPS), RNG.uniform(0, 1.5, size=N_TRIPS))
pressure_end_psi = np.round(pressure_start_psi + heat_gain - leak_loss, 1)

# ---- Assemble DataFrame ----
df = pd.DataFrame({
    "trip_id": trip_id,
    "cost_usd": cost_usd,
    "distance_km": distance_km,
    "tread_depth_mm": tread_depth_mm,
    "has_cracks": has_cracks,
    "pressure_start_psi": pressure_start_psi,
    "pressure_end_psi": pressure_end_psi,
})

# ---- Save to CSV ----
output_path = "synthetic.csv"
df.to_csv(output_path, index=False)

print(f"Saved {len(df)} rows to {output_path}")
print(df.head())

Saved 2000 rows to synthetic.csv
   trip_id  cost_usd  distance_km  tread_depth_mm  has_cracks  \
0        1    285.89        275.7           12.02       False   
1        2    392.13        347.2           17.36       False   
2        3    238.73        250.5            8.57        True   
3        4    213.40        231.1           17.56       False   
4        5    411.72        370.1           12.52        True   

   pressure_start_psi  pressure_end_psi  
0               109.2             114.2  
1               108.3             112.7  
2               107.5             109.6  
3               104.2             108.2  
4               114.1             114.6  


In [2]:
data = pd.read_csv("synthetic.csv")
data

,trip_id,cost_usd,distance_km,tread_depth_mm,has_cracks,pressure_start_psi,pressure_end_psi
0,1,285.89,275.7,12.02,False,109.2,114.2
1,2,392.13,347.2,17.36,False,108.3,112.7
2,3,238.73,250.5,8.57,True,107.5,109.6
3,4,213.40,231.1,17.56,False,104.2,108.2
4,5,411.72,370.1,12.52,True,114.1,114.6
...,...,...,...,...,...,...,...
1995,1996,205.21,271.1,9.56,False,118.3,119.4
1996,1997,421.13,479.9,13.08,False,112.2,119.3
1997,1998,191.40,194.4,10.88,False,118.5,118.8
1998,1999,119.44,141.5,9.94,False,109.6,111.9
